# Speed profiles

Constrain speed by curvature, acceleration and braking distance.

Run with the repository virtual environment. These are synthetic research experiments; units and assumptions are explicit below.

## Reference path

A sampled circle supplies a known curvature scale.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from autonomous_vehicles.planning.path import path_from_xy
from autonomous_vehicles.planning.speed import (
    SpeedProfileConfig,
    plan_speed,
    trajectory_from_profile,
)

angles = np.linspace(0, np.pi / 2, 101)
positions = np.column_stack((20 * np.sin(angles), 20 * (1 - np.cos(angles))))
path = path_from_xy(positions)
config = SpeedProfileConfig(maximum_speed=10, lateral_acceleration=2)
profile = plan_speed(path, config=config)

## Forward and backward passes

The profile respects both acceleration from the start and braking toward the endpoint.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(profile.distances, profile.speeds)
axes[0].set_xlabel("Arc length (m)")
axes[0].set_ylabel("Speed (m/s)")
axes[1].plot(profile.times, profile.speeds)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Speed (m/s)")
plt.show()
print("Traversal time (s):", profile.duration)

## Verify constraints

Use the energy-form acceleration relation and v² times curvature.

In [ ]:
longitudinal = np.diff(profile.speeds**2) / (2 * np.diff(profile.distances))
curvature = np.array([point.curvature for point in path.points])
lateral = profile.speeds**2 * np.abs(curvature)
assert longitudinal.max() <= config.acceleration + 1e-9
assert longitudinal.min() >= -config.deceleration - 1e-9
assert lateral.max() <= config.lateral_acceleration + 1e-9
print("Peak acceleration:", longitudinal.max())
print("Peak deceleration:", -longitudinal.min())
print("Peak lateral acceleration:", lateral.max())

## Interpolate in time

The trajectory linearly interpolates pose and speed between profile samples.

In [ ]:
trajectory = trajectory_from_profile(path, profile)
query_times = np.linspace(0, trajectory.duration, 40)
samples = [trajectory.sample(float(time)) for time in query_times]
fig, ax = plt.subplots(figsize=(6, 5))
scatter = ax.scatter(
    [p.pose.x for p in samples],
    [p.pose.y for p in samples],
    c=[p.speed for p in samples],
    cmap="viridis",
)
fig.colorbar(scatter, ax=ax, label="Speed (m/s)")
ax.set_aspect("equal")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
plt.show()

## Sensitivity

The same geometric path can yield different traversal times under comfort limits.

In [ ]:
limits = (0.5, 1.0, 2.0, 3.0)
durations = []
for limit in limits:
    candidate = plan_speed(path, config=SpeedProfileConfig(lateral_acceleration=limit))
    durations.append(candidate.duration)
fig, ax = plt.subplots()
ax.plot(limits, durations, marker="o")
ax.set_xlabel("Allowed lateral acceleration (m/s²)")
ax.set_ylabel("Traversal time (s)")
plt.show()

## Research record

Record the seed, configuration, dependency lock and observed failures with each experiment. Results here demonstrate the implemented model under its stated assumptions. They do not establish performance outside this synthetic distribution.